### `Proposta do notebook:`

Avaliar o uso de um multilayer perceptron nas variáveis categóricas e numéricas.

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
import torch.optim as optim


seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando o device: {device}")

Usando o device: cpu


In [3]:


# 1. Carregar os dados
df = pd.read_csv("../data/cubagem_40k_amazon.csv")

# Separar splits
df_train = df[df['split'] == 'train'].copy()
df_val = df[df['split'] == 'val'].copy()

# ---------------------------------------------------------
# A. TRATAMENTO DAS VARIÁVEIS NUMÉRICAS
# ---------------------------------------------------------
colunas_numericas = [
    'price_numeric', 'avg_rating', 'n_ratings', 'n_images', 
    'title_length', 'title_word_count', 'description_length', 'n_categories_levels'
]

# Preencher buracos com a mediana do treino
for col in colunas_numericas:
    mediana = df_train[col].median()
    # Se uma coluna inteira for NaN, a mediana será NaN. Vamos forçar 0 como fallback
    if pd.isna(mediana): mediana = 0  
    
    df_train[col] = df_train[col].fillna(mediana)
    df_val[col] = df_val[col].fillna(mediana)

# Escalonar numéricas
scaler_num = StandardScaler()
X_train_num = scaler_num.fit_transform(df_train[colunas_numericas])
X_val_num = scaler_num.transform(df_val[colunas_numericas])

# ---------------------------------------------------------
# B. TRATAMENTO DAS VARIÁVEIS CATEGÓRICAS (LABEL ENCODING)
# ---------------------------------------------------------
colunas_categoricas = ['main_category', 'source_category', 'categories', 'store']

# Preencher NaNs com a string "Desconhecido"
for col in colunas_categoricas:
    df_train[col] = df_train[col].fillna("Desconhecido").astype(str)
    df_val[col] = df_val[col].fillna("Desconhecido").astype(str)

# Dicionário para guardar as informações de tamanho de cada Embedding
embedding_sizes = {}
encoders = {}
X_train_cat = np.zeros((len(df_train), len(colunas_categoricas)), dtype=np.int64)
X_val_cat = np.zeros((len(df_val), len(colunas_categoricas)), dtype=np.int64)

for i, col in enumerate(colunas_categoricas):
    le = LabelEncoder()
    # Ajustar o encoder combinando treino e validação para evitar classes invisíveis na validação
    todos_os_valores = pd.concat([df_train[col], df_val[col]]).unique()
    le.fit(todos_os_valores)
    
    X_train_cat[:, i] = le.transform(df_train[col])
    X_val_cat[:, i] = le.transform(df_val[col])
    
    encoders[col] = le
    
    # Calcular tamanho do embedding: min(50, num_classes // 2)
    num_classes = len(le.classes_)
    emb_dim = min(50, max(1, num_classes // 2))
    embedding_sizes[col] = (num_classes, emb_dim)

print("Tamanhos dos Embeddings (Num_Classes, Dimensão_Vetor):", embedding_sizes)

# ---------------------------------------------------------
# C. TRATAMENTO DO TARGET (CUBAGEM)
# ---------------------------------------------------------
colunas_alvo = ['length_cm', 'width_cm', 'height_cm', 'weight_g']
df_train = df_train.dropna(subset=colunas_alvo)
df_val = df_val.dropna(subset=colunas_alvo)

scaler_target = StandardScaler()
y_train = scaler_target.fit_transform(df_train[colunas_alvo])
y_val = scaler_target.transform(df_val[colunas_alvo])

Tamanhos dos Embeddings (Num_Classes, Dimensão_Vetor): {'main_category': (44, 22), 'source_category': (7, 3), 'categories': (4407, 50), 'store': (25614, 50)}


In [4]:
class TabularCubagemDataset(Dataset):
    def __init__(self, cat_features, num_features, targets):
        self.cat_features = torch.tensor(cat_features, dtype=torch.long)
        self.num_features = torch.tensor(num_features, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return {
            'cat_features': self.cat_features[idx],
            'num_features': self.num_features[idx],
            'targets': self.targets[idx]
        }

dataset_train = TabularCubagemDataset(X_train_cat, X_train_num, y_train)
dataset_val = TabularCubagemDataset(X_val_cat, X_val_num, y_val)

train_loader = DataLoader(dataset_train, batch_size=128, shuffle=True)
val_loader = DataLoader(dataset_val, batch_size=128, shuffle=False)

In [15]:
class MLPTabular(nn.Module):
    def __init__(self, embedding_sizes, num_numerical_features, num_targets=4):
        super(MLPTabular, self).__init__()
        
        # Cria as camadas de Embedding usando a lista calculada anteriormente
        # nn.ModuleList é essencial para o PyTorch reconhecer essas camadas internas
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_classes, emb_dim) for num_classes, emb_dim in embedding_sizes.values()
        ])
        
        # Calcula o tamanho total de todos os vetores de embedding somados
        total_emb_dim = sum(emb_dim for _, emb_dim in embedding_sizes.values())
        
        # O tamanho de entrada da nossa rede = embeddings + colunas numéricas
        input_size = total_emb_dim + num_numerical_features
        
        # Arquitetura da Rede Neural
        self.mlp = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            
            nn.Linear(256, 128),
            nn.ReLU(),
            
            nn.Linear(128, num_targets) # Sai as 4 dimensões
        )

    def forward(self, x_cat, x_num):
        # x_cat tem shape (batch_size, num_categorical_columns)
        emb_outputs = []
        
        # Passa cada coluna categórica pelo seu respectivo Embedding
        for i, emb_layer in enumerate(self.embeddings):
            # Pega a coluna i (todas as linhas do batch) e passa no embedding
            emb_outputs.append(emb_layer(x_cat[:, i]))
            
        # Concatena todos os embeddings em um único vetor longo
        x_cat_embedded = torch.cat(emb_outputs, dim=1)
        
        # Concatena os embeddings com os números auxiliares
        x_final = torch.cat([x_cat_embedded, x_num], dim=1)
        
        # Passa tudo pela MLP principal
        return self.mlp(x_final)

# Instanciando o modelo
modelo = MLPTabular(
    embedding_sizes=embedding_sizes, 
    num_numerical_features=len(colunas_numericas)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo.to(device)
print(modelo)

MLPTabular(
  (embeddings): ModuleList(
    (0): Embedding(44, 22)
    (1): Embedding(7, 3)
    (2): Embedding(4407, 50)
    (3): Embedding(25614, 50)
  )
  (mlp): Sequential(
    (0): Linear(in_features=133, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.5, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): ReLU()
    (10): Linear(in_features=128, out_features=4, bias=True)
  )
)


In [16]:

# ---------------------------------------------------------
# 4. LOOP DE TREINAMENTO E VALIDAÇÃO
# ---------------------------------------------------------
criterion = nn.HuberLoss()
# Passamos todos os parâmetros da nossa rede tabular para o otimizador
optimizer = optim.Adam(modelo.parameters(), lr=1e-3, weight_decay=1e-5) # weight_decay ajuda a evitar overfitting

epocas = 10 # Modelos tabulares treinam rápido, podemos colocar mais épocas

for epoch in range(epocas):
    
    # ==========================================
    # FASE 1: TREINAMENTO
    # ==========================================
    modelo.train() # Ativa Dropout e permite que o BatchNorm calcule as médias do batch
    train_loss = 0
    
    for batch in train_loader:
        # Puxamos as features separadas do dicionário
        x_cat = batch['cat_features'].to(device)
        x_num = batch['num_features'].to(device)
        targets_batch = batch['targets'].to(device)
        
        optimizer.zero_grad()
        
        # O forward agora recebe as duas variáveis
        predicoes = modelo(x_cat, x_num)
        
        loss = criterion(predicoes, targets_batch)
        loss.backward()
        
        optimizer.step()
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_loader)

    # ==========================================
    # FASE 2: VALIDAÇÃO E VISUALIZAÇÃO
    # ==========================================
    modelo.eval() # Desliga Dropout e trava as médias do BatchNorm
    val_loss = 0
    exemplos_capturados = False 
    
    with torch.no_grad():
        for batch in val_loader:
            x_cat = batch['cat_features'].to(device)
            x_num = batch['num_features'].to(device)
            targets_batch = batch['targets'].to(device)
            
            predicoes = modelo(x_cat, x_num)
            loss = criterion(predicoes, targets_batch)
            val_loss += loss.item()
            
            # Captura o primeiro batch para mostrar exemplos
            if not exemplos_capturados:
                preds_np = predicoes.cpu().numpy()
                targets_np = targets_batch.cpu().numpy()
                
                # Reverte a escala (StandardScaler) para visualizar cm e gramas
                preds_reais = scaler_target.inverse_transform(preds_np)
                targets_reais = scaler_target.inverse_transform(targets_np)
                
                exemplos_capturados = True
                
    avg_val_loss = val_loss / len(val_loader)
    
    # ==========================================
    # FASE 3: EXIBIÇÃO DE RESULTADOS
    # ==========================================
    print(f"\n[Época {epoch+1}/{epocas}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print("-" * 60)
    print("Visualizando predições (Validação):")
    
    # Mostra os 3 primeiros exemplos capturados
    for i in range(min(3, len(preds_reais))):
        print(f"  > Exemplo {i+1}:")
        print(f"    Real -> Comp: {targets_reais[i][0]:.1f}cm | Larg: {targets_reais[i][1]:.1f}cm | Alt: {targets_reais[i][2]:.1f}cm | Peso: {targets_reais[i][3]:.1f}g")
        print(f"    Pred -> Comp: {preds_reais[i][0]:.1f}cm | Larg: {preds_reais[i][1]:.1f}cm | Alt: {preds_reais[i][2]:.1f}cm | Peso: {preds_reais[i][3]:.1f}g")
    print("=" * 60)


[Época 1/10] Train Loss: 0.2481 | Val Loss: 0.2325
------------------------------------------------------------
Visualizando predições (Validação):
  > Exemplo 1:
    Real -> Comp: 56.4cm | Larg: 7.9cm | Alt: 7.6cm | Peso: 249.5g
    Pred -> Comp: 26.3cm | Larg: 15.0cm | Alt: 6.9cm | Peso: 797.7g
  > Exemplo 2:
    Real -> Comp: 7.0cm | Larg: 7.0cm | Alt: 1.3cm | Peso: 64.1g
    Pred -> Comp: 19.3cm | Larg: 11.6cm | Alt: 4.7cm | Peso: 380.4g
  > Exemplo 3:
    Real -> Comp: 26.7cm | Larg: 8.9cm | Alt: 8.9cm | Peso: 544.3g
    Pred -> Comp: 26.9cm | Larg: 16.2cm | Alt: 7.6cm | Peso: 879.9g

[Época 2/10] Train Loss: 0.2307 | Val Loss: 0.2210
------------------------------------------------------------
Visualizando predições (Validação):
  > Exemplo 1:
    Real -> Comp: 56.4cm | Larg: 7.9cm | Alt: 7.6cm | Peso: 249.5g
    Pred -> Comp: 26.6cm | Larg: 15.8cm | Alt: 7.0cm | Peso: 745.0g
  > Exemplo 2:
    Real -> Comp: 7.0cm | Larg: 7.0cm | Alt: 1.3cm | Peso: 64.1g
    Pred -> Comp: 19.2cm